# 4. Artifacts, computation cache, retention, and garbage collection

Provium still owns artifact format, identity, metadata, and lineage. Pipeline owns the operational problem of keeping finalized artifacts somewhere durable, locating them later, materializing them for workers, publishing outputs, and eventually reclaiming storage.


## Descriptor, location, store, and index

A `ManagedArtifactDescriptor` is the logical artifact facts. An `ArtifactLocation` says where one physical copy lives and its state. An `ArtifactStore` moves bytes; an `ArtifactIndex` maps identities to locations. Separating these contracts permits multiple stores, replicas, repair, and remote adapters without changing artifact identity.


In [ ]:
from provium_pipeline.artifact import (
    ArtifactImportService,
    ArtifactIndex,
    ArtifactLocation,
    ArtifactStore,
    FilesystemArtifactStore,
    ManagedArtifactDescriptor,
    SQLiteArtifactIndex,
)

artifact_layers = {
    'logical': ManagedArtifactDescriptor, 'physical': ArtifactLocation,
    'store_contract': ArtifactStore, 'index_contract': ArtifactIndex,
    'local_store': FilesystemArtifactStore, 'local_index': SQLiteArtifactIndex,
    'import_boundary': ArtifactImportService,
}
assert all(artifact_layers.values())
artifact_layers


## Import, materialization, and publication

Import inspects a finalized `.pa`, derives its identity, stages bytes, verifies them, promotes them atomically, and then registers the location. Materialization creates a task-local readable path plus an explicit cleanup handle. Publication records the immutable mapping from a task output field to an artifact identity; a conflicting second publication is an error, while the exact same publication is idempotent.


## Computation cache is not artifact storage

The computation key covers the procedure contract, resolved configuration, and input artifact identities. A cache entry points to already published output identities. Reservation prevents duplicate work; waiters may reuse a completed entry, while expired reservations can be recovered. The artifacts themselves remain in stores and must still be retained.


In [ ]:
from provium_pipeline.cache.store import (
    CacheReservationDisposition,
    ComputationCacheEntry,
    InMemoryComputationCache,
    InvalidCacheEntryError,
)
from provium_pipeline.lifecycle.retention import (
    InMemoryRetentionIndex,
    RetentionClass,
    RetentionReference,
)

assert CacheReservationDisposition.OWNER.value == 'owner'
assert all((ComputationCacheEntry, InMemoryComputationCache, InvalidCacheEntryError))
assert all((InMemoryRetentionIndex, RetentionClass, RetentionReference))


## Retention and garbage collection

Retention references answer *why must this artifact remain?* Sources include input sets, run snapshots, task outputs, and cache entries. Garbage collection computes unretained candidates older than a safety cutoff, creates a reviewable plan, deletes through the store, and records per-object results. It is deliberately a two-step safety workflow.

An **orphan** is different: bytes exist in a store without a valid index location, often after a crash between file promotion and registration. Store orphan scans reconcile that physical inconsistency. An indexed but unretained artifact is a normal GC candidate.


In [ ]:
from provium_pipeline.lifecycle.garbage_collection import (
    GarbageCollectionPlan,
    GarbageCollectionResult,
    GarbageCollectionService,
)

gc_contract = (GarbageCollectionService, GarbageCollectionPlan, GarbageCollectionResult)
assert all(gc_contract)
gc_contract


**What to notice:** storage manages bytes, the index manages locations, publication manages semantic ownership, retention protects live identities, GC reclaims unprotected identities, and orphan cleanup repairs store/index disagreement. Next: [storage and recovery](05-storage-recovery-and-concurrency.ipynb). Reference: [artifacts](../docs/artifacts.md).
